[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/dcf_inputs.ipynb)

# DCF Inputs — Build Your Own Two-Stage DCF

Build the raw inputs for a two-stage Discounted Cash Flow model from the `fact` table, then cross-check against Valuein's pre-computed `valuation` table.

This is the notebook to run when you want to roll your own DCF — we hand you the ingredients, you choose the WACC.

---

In [ ]:
!pip install valuein-sdk -q

In [ ]:
try:
    import os

    from google.colab import userdata

    os.environ["VALUEIN_API_KEY"] = userdata.get("VALUEIN_API_KEY")
except ImportError:
    pass

In [ ]:
from valuein_sdk import ValueinClient

TICKER = "MSFT"
client = ValueinClient(tables=["references", "filing", "fact", "valuation"])

## 1. Five-year annual FCF history

CAPEX sign varies by filer — always wrap in `ABS()` before subtracting from Operating Cash Flow.

In [ ]:
fcf = client.query(f"""
    WITH fy_10k AS (
        SELECT
            fa.accession_id, fa.fiscal_year, fa.period_end,
            MAX(CASE WHEN fa.standard_concept = 'OperatingCashFlow' THEN fa.numeric_value END) AS ocf,
            MAX(CASE WHEN fa.standard_concept = 'CAPEX'             THEN fa.numeric_value END) AS capex
        FROM fact fa
        JOIN filing  f  ON fa.accession_id = f.accession_id
        JOIN "references" r ON f.entity_id    = r.cik
        WHERE r.symbol        = '{TICKER}'
          AND r.is_active     = TRUE
          AND f.form_type     = '10-K'
          AND fa.fiscal_period = 'FY'
          AND fa.standard_concept IN ('OperatingCashFlow', 'CAPEX')
        GROUP BY fa.accession_id, fa.fiscal_year, fa.period_end
        QUALIFY ROW_NUMBER() OVER (PARTITION BY fa.fiscal_year ORDER BY fa.period_end DESC) = 1
    )
    SELECT
        fiscal_year,
        period_end,
        round(ocf / 1e9, 3)                 AS ocf_bn,
        round(ABS(capex) / 1e9, 3)          AS capex_bn,
        round((ocf - ABS(capex)) / 1e9, 3)  AS fcf_bn
    FROM fy_10k
    WHERE ocf IS NOT NULL AND capex IS NOT NULL
    ORDER BY fiscal_year DESC
    LIMIT 5
""")
fcf

## 2. Stage-1 growth CAGR from the FCF series

In [ ]:
if len(fcf) >= 2:
    oldest = fcf["fcf_bn"].iloc[-1]
    newest = fcf["fcf_bn"].iloc[0]
    years = len(fcf) - 1
    if oldest and oldest > 0 and newest > 0:
        cagr = (newest / oldest) ** (1 / years) - 1
        print(
            f"{years}-year FCF CAGR: {cagr:.2%}  (Stage-1 growth assumption starting point)"
        )
    else:
        print("CAGR not computable (negative or zero FCF in window)")
else:
    print("Not enough FCF history in the Sample tier for a CAGR")

## 3. Latest balance sheet context

A DCF needs the capital structure to convert enterprise value → equity value, and EPS_Diluted (or share count) to convert equity value → per-share value.

In [ ]:
bs = client.query(f"""
    WITH latest AS (
        SELECT f.accession_id
        FROM   filing    f
        JOIN   "references" r ON f.entity_id = r.cik
        WHERE  r.symbol    = '{TICKER}'
          AND  r.is_active = TRUE
          AND  f.form_type = '10-K'
        ORDER  BY f.filing_date DESC
        LIMIT  1
    )
    SELECT
        fa.standard_concept,
        round(fa.numeric_value / 1e9, 3) AS value_bn,
        fa.period_end
    FROM fact fa
    JOIN latest l ON fa.accession_id = l.accession_id
    WHERE fa.standard_concept IN (
        'CashAndEquivalents',
        'TotalAssets',
        'TotalLiabilities',
        'StockholdersEquity',
        'EPS_Diluted'
    )
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY fa.standard_concept ORDER BY fa.period_end DESC
    ) = 1
    ORDER BY fa.standard_concept
""")
bs

## 4. Valuein pre-computed DCF (cross-check)

The `valuation` table contains two-stage DCF + DDM intrinsic values keyed by entity + period_end. Useful as a sanity check on your own model.

In [ ]:
val = client.query(f"""
    SELECT
        v.period_end,
        round(v.dcf_value_per_share, 2)      AS dcf_per_share,
        round(v.ddm_value_per_share, 2)      AS ddm_per_share,
        round(v.wacc, 4)                     AS wacc,
        round(v.terminal_growth_rate, 4)     AS terminal_g,
        round(v.stage1_growth_rate, 4)       AS stage1_g,
        v.stage1_years                       AS stage1_yrs,
        round(v.fcf_base / 1e9, 3)           AS fcf_base_bn,
        round(v.shares_outstanding / 1e9, 3) AS shares_bn
    FROM   valuation v
    JOIN   "references" r ON v.entity_id = r.cik
    WHERE  r.symbol    = '{TICKER}'
      AND  r.is_active = TRUE
    ORDER  BY v.period_end DESC
    LIMIT  3
""")
val

## Next steps

You now have: FCF history, balance sheet, WACC + terminal growth. Drop these into your own two-stage DCF sheet (or `valuein_sdk.dcf.*` helpers when you upgrade) to produce a per-share intrinsic value.

- [`factor_screen.ipynb`](factor_screen.ipynb) — rank the universe by Quality / Growth / Efficiency
- [`earnings_momentum.ipynb`](earnings_momentum.ipynb) — spot accelerating vs decelerating fundamentals